# BFP Digital Twin & Performance Tracking

From the [Sisyphean Gridworks ML Playground](https://sgridworks.com/ml-playground/guides/20-bfp-digital-twin.html)

## Setup

Clone the repository and install dependencies. Run this cell first.

In [ ]:
import os, subprocess

# Colab: clone the repo and cd into it
# Local: detect if we are already inside the repo
if not os.path.exists('sisyphean-power-and-light'):
    if os.path.exists('../sisyphean-power-and-light'):
        os.chdir('..')  # running from notebooks/ subfolder
    else:
        subprocess.run(['git', 'clone', 'https://github.com/SGridworks/Dynamic-Network-Model.git'], capture_output=True)
        os.chdir('Dynamic-Network-Model')

print(f'Working directory: {os.getcwd()}')
# !pip install -q pandas numpy matplotlib seaborn scikit-learn pyarrow scipy


## Step 0: Load All Data Sources

A digital twin combines physics (OEM pump curves) with data (actual operating measurements). We load three sources:
1. **Pump curves** -- the manufacturer's head-flow and efficiency-flow curves (what the pump *should* do)
2. **Hourly operating data** -- what the pump *actually* did
3. **Operator actions** -- when swaps and shutdowns happened

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.interpolate import interp1d

# Load pump curves (OEM reference)
pump_curves = pd.read_csv(
    "sisyphean-power-and-light/generation/reference/pump_curves.csv"
)
print(f"Pump curve points: {len(pump_curves)}")
print(pump_curves.head())

# Load BFP hourly operating data
df = pd.read_parquet(
    "sisyphean-power-and-light/generation/timeseries/bfp_train_hourly.parquet"
)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()
print(f"\nOperating data: {len(df):,} rows")

# Load design parameters
with open("sisyphean-power-and-light/generation/reference/design_parameters.json") as f:
    design = json.load(f)

# Load operator actions
actions = pd.read_csv(
    "sisyphean-power-and-light/generation/events/operator_actions.csv"
)
actions["timestamp"] = pd.to_datetime(actions["timestamp"])
print(f"\nOperator actions: {len(actions)}")
print(actions.to_string(index=False))

assert len(df) == 8784, f"Expected 8,784 rows, got {len(df)}"
print("\nSetup verified.")

## Step 1: Plot OEM Pump Curves

The pump curves define the manufacturer's guaranteed performance at rated speed. These are the "physics" component of our digital twin. The key curves are:
- **Head-Flow**: how much differential pressure the pump produces at each flow rate
- **Efficiency-Flow**: how efficiently the pump converts motor power to hydraulic power
- **Power-Flow**: the mechanical power required at each flow rate

In [ ]:
# Plot OEM pump curves
fig, axes = plt.subplots(1, 3, figsize=(10, 5))

# Head-Flow curve
axes[0].plot(pump_curves["flow_tph"], pump_curves["head_bar"],
             color="#2D6A7A", linewidth=2)
axes[0].set_xlabel("Flow (t/hr)")
axes[0].set_ylabel("Head (bar)")
axes[0].set_title("Head-Flow Curve")
axes[0].axvline(x=design["bfp"]["bep_flow_tph"], color="#D69E2E",
                linestyle="--", alpha=0.7, label=f"BEP ({design['bfp']['bep_flow_tph']} t/hr)")
axes[0].legend(fontsize=8)

# Efficiency-Flow curve
axes[1].plot(pump_curves["flow_tph"], pump_curves["efficiency_pct"],
             color="#5FCCDB", linewidth=2)
axes[1].set_xlabel("Flow (t/hr)")
axes[1].set_ylabel("Efficiency (%)")
axes[1].set_title("Efficiency-Flow Curve")
axes[1].axvline(x=design["bfp"]["bep_flow_tph"], color="#D69E2E",
                linestyle="--", alpha=0.7, label="BEP")
axes[1].axhline(y=design["bfp"]["bep_efficiency"] * 100, color="gray",
                linestyle=":", alpha=0.5)
axes[1].legend(fontsize=8)

# Power-Flow curve
axes[2].plot(pump_curves["flow_tph"], pump_curves["power_kw"],
             color="#E53E3E", linewidth=2)
axes[2].set_xlabel("Flow (t/hr)")
axes[2].set_ylabel("Power (kW)")
axes[2].set_title("Power-Flow Curve")
axes[2].axvline(x=design["bfp"]["bep_flow_tph"], color="#D69E2E",
                linestyle="--", alpha=0.7, label="BEP")
axes[2].legend(fontsize=8)

plt.suptitle("OEM Pump Curves (KSB CHTD 8/6 at rated speed)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"Best Efficiency Point (BEP): {design['bfp']['bep_flow_tph']} t/hr")
print(f"BEP Efficiency: {design['bfp']['bep_efficiency'] * 100}%")
print(f"Rated flow: {design['bfp']['rated_flow_tph']} t/hr")

## Step 2: Compute Expected Performance from Pump Curves

For each operating hour, we look up the actual flow on the OEM curve and interpolate to find the **expected** power consumption. The difference between actual and expected power is the performance deviation.

In [ ]:
# Create interpolation functions from OEM curves
# Exclude zero-flow point for interpolation stability
pc = pump_curves[pump_curves["flow_tph"] > 0].copy()

f_head = interp1d(pc["flow_tph"], pc["head_bar"],
                  kind="cubic", fill_value="extrapolate")
f_eff  = interp1d(pc["flow_tph"], pc["efficiency_pct"],
                  kind="cubic", fill_value="extrapolate")
f_power = interp1d(pc["flow_tph"], pc["power_kw"],
                   kind="cubic", fill_value="extrapolate")

# Identify active pump and create unified columns
df["active_pump"] = "none"
df.loc[df["U1_BFPA_RUN_STATUS"] > 0, "active_pump"] = "A"
df.loc[df["U1_BFPB_RUN_STATUS"] > 0, "active_pump"] = "B"

running = df[df["active_pump"] != "none"].copy()

running["flow"] = np.where(running["active_pump"] == "A",
                           running["U1_BFPA_FW_FLOW"], running["U1_BFPB_FW_FLOW"])
running["power_actual"] = np.where(running["active_pump"] == "A",
                                    running["U1_BFPA_MTR_POWER"], running["U1_BFPB_MTR_POWER"])
running["dp_actual"] = np.where(running["active_pump"] == "A",
                                 running["U1_BFPA_DISCH_PRESS"], running["U1_BFPB_DISCH_PRESS"])

# Compute expected values from OEM curves
running["power_expected"] = f_power(running["flow"].clip(lower=13, upper=338))
running["head_expected"] = f_head(running["flow"].clip(lower=13, upper=338))
running["eff_expected"] = f_eff(running["flow"].clip(lower=13, upper=338))

# Performance deviations
running["power_deviation"] = running["power_actual"] - running["power_expected"]
running["head_deviation"] = running["dp_actual"] - running["head_expected"]

print(f"Running hours with OEM comparison: {len(running):,}")
print(f"\nPower deviation stats:")
print(f"  Mean: {running['power_deviation'].mean():.1f} kW")
print(f"  Std:  {running['power_deviation'].std():.1f} kW")
print(f"  Min:  {running['power_deviation'].min():.1f} kW")
print(f"  Max:  {running['power_deviation'].max():.1f} kW")

## Step 3: Overlay Actual Operating Points on OEM Curves

Plot the actual operating points on top of the OEM curves. In a healthy pump, actual points should cluster tightly along the OEM curve. During faults, they drift away -- this drift is the digital twin's primary diagnostic signal.

In [ ]:
# Overlay actual data on OEM head-flow and power-flow curves
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Head-Flow
axes[0].plot(pump_curves["flow_tph"], pump_curves["head_bar"],
             color="#2D6A7A", linewidth=2, label="OEM Curve", zorder=5)

# Color by period: healthy (Jan-Mar), seal (Apr-Jun), bearing (Jul-Aug), misalignment (Oct-Dec)
periods = [
    ("2024-01-01", "2024-04-01", "#5FCCDB", "Healthy (Jan-Mar)"),
    ("2024-04-01", "2024-06-01", "#D69E2E", "Seal fault (Apr-May)"),
    ("2024-06-01", "2024-08-21", "#E53E3E", "Bearing (Jun-Aug)"),
    ("2024-09-15", "2024-10-15", "#5FCCDB", "Post-outage healthy"),
    ("2024-10-15", "2024-12-31", "#718096", "Misalignment (Oct-Dec)"),
]

for start, end, color, label in periods:
    mask = (running.index >= start) & (running.index < end)
    axes[0].scatter(running.loc[mask, "flow"], running.loc[mask, "dp_actual"],
                    c=color, s=2, alpha=0.3, label=label)
    axes[1].scatter(running.loc[mask, "flow"], running.loc[mask, "power_actual"],
                    c=color, s=2, alpha=0.3, label=label)

axes[0].set_xlabel("Flow (t/hr)")
axes[0].set_ylabel("Discharge Pressure (bar)")
axes[0].set_title("Actual vs OEM: Head-Flow")
axes[0].legend(fontsize=7, markerscale=5)

# Power-Flow
axes[1].plot(pump_curves["flow_tph"], pump_curves["power_kw"],
             color="#2D6A7A", linewidth=2, label="OEM Curve", zorder=5)
axes[1].set_xlabel("Flow (t/hr)")
axes[1].set_ylabel("Motor Power (kW)")
axes[1].set_title("Actual vs OEM: Power-Flow")
axes[1].legend(fontsize=7, markerscale=5)

plt.tight_layout()
plt.show()

## Step 4: Plot Performance Deviation Over Time

The deviation between actual and expected power is the efficiency decay signal. Positive deviation means the pump is consuming more power than expected for the given flow -- it is losing efficiency. This is the signature of internal degradation (seal wear, increased clearances, fouling).

In [ ]:
# Power deviation over time with 24h rolling mean
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

roll_power = running["power_deviation"].rolling(24, min_periods=1).mean()
roll_head  = running["head_deviation"].rolling(24, min_periods=1).mean()

axes[0].plot(running.index, running["power_deviation"],
             color="#5FCCDB", linewidth=0.3, alpha=0.4)
axes[0].plot(running.index, roll_power,
             color="#2D6A7A", linewidth=1.5, label="24h rolling mean")
axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[0].set_ylabel("Power Deviation (kW)")
axes[0].set_title("Power Deviation: Actual - OEM Expected (positive = efficiency loss)")
axes[0].legend(fontsize=8)

# Shade fault periods
for ax in axes:
    ax.axvspan(pd.Timestamp("2024-04-01"), pd.Timestamp("2024-06-01"),
               alpha=0.1, color="#D69E2E")
    ax.axvspan(pd.Timestamp("2024-07-15"), pd.Timestamp("2024-08-21"),
               alpha=0.1, color="#E53E3E")
    ax.axvspan(pd.Timestamp("2024-10-15"), pd.Timestamp("2024-12-31"),
               alpha=0.1, color="#718096")

axes[1].plot(running.index, running["head_deviation"],
             color="#5FCCDB", linewidth=0.3, alpha=0.4)
axes[1].plot(running.index, roll_head,
             color="#2D6A7A", linewidth=1.5, label="24h rolling mean")
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
axes[1].set_ylabel("Head Deviation (bar)")
axes[1].set_xlabel("Date")
axes[1].set_title("Head Deviation: Actual - OEM Expected")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Step 5: Physics-Informed ML Model -- Train on Healthy Baseline

The OEM curves are idealized. In practice, even a healthy pump operates slightly differently due to installation effects, piping losses, and instrument calibration offsets. A physics-informed model learns these site-specific corrections by training on healthy data (Jan-Mar) only. Going forward, deviations from this model represent genuine degradation, not calibration offsets.

In [ ]:
# Physics-informed features: OEM expected values + operating conditions
running["oem_power"] = running["power_expected"]
running["oem_head"] = running["head_expected"]
running["oem_eff"] = running["eff_expected"]

# Features for the baseline model
baseline_features = [
    "flow", "U1_UNIT_MW_GROSS", "U1_AMBIENT_TEMP",
    "oem_power", "oem_head", "oem_eff",
]

# Targets: actual power, actual head, actual vibration
targets = {
    "power": "power_actual",
    "head": "dp_actual",
}

# Train on healthy period (Jan-Mar, BFP-A only)
healthy = running[
    (running.index < "2024-04-01") &
    (running["active_pump"] == "A")
].copy()

models = {}
for name, target_col in targets.items():
    X = healthy[baseline_features].fillna(0)
    y = healthy[target_col]
    model = GradientBoostingRegressor(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        random_state=42
    )
    model.fit(X, y)
    y_pred = model.predict(X)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    models[name] = model
    print(f"Baseline model ({name}):")
    print(f"  R-squared: {r2:.4f}")
    print(f"  MAE: {mae:.2f}")
    print()

## Step 6: Residual Monitoring -- Physics-Informed Predictions vs Actual

Apply the baseline models to the entire year. Residuals (actual minus predicted) should be near zero during healthy periods and grow during fault periods. This is the digital twin's predictive power: the model "knows" what a healthy pump should produce at any operating point.

In [ ]:
# Predict for all running data
X_all = running[baseline_features].fillna(0)

for name, model in models.items():
    running[f"{name}_predicted"] = model.predict(X_all)
    running[f"{name}_residual"] = running[targets[name]] - running[f"{name}_predicted"]

# Plot residuals over time
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

for ax, name, ylabel in zip(axes,
    ["power", "head"],
    ["Power Residual (kW)", "Head Residual (bar)"]):

    resid = running[f"{name}_residual"]
    roll = resid.rolling(24, min_periods=1).mean()

    ax.plot(running.index, resid, color="#5FCCDB", linewidth=0.3, alpha=0.4)
    ax.plot(running.index, roll, color="#2D6A7A", linewidth=1.5,
            label="24h rolling mean")
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)

    # Shade fault periods
    ax.axvspan(pd.Timestamp("2024-04-01"), pd.Timestamp("2024-06-01"),
               alpha=0.1, color="#D69E2E")
    ax.axvspan(pd.Timestamp("2024-07-15"), pd.Timestamp("2024-08-21"),
               alpha=0.1, color="#E53E3E")
    ax.axvspan(pd.Timestamp("2024-10-15"), pd.Timestamp("2024-12-31"),
               alpha=0.1, color="#718096")

axes[0].set_title("Physics-Informed Model Residuals (actual - baseline predicted)")
axes[-1].set_xlabel("Date")

plt.tight_layout()
plt.show()

# Monthly residual statistics
running["month"] = running.index.month
print("Monthly Power Residual (kW):")
print(running.groupby("month")["power_residual"].agg(["mean", "std"]).round(1).to_string())

## Step 7: Create a Pump Health Index (0-100)

Individual residuals are useful for engineers, but operations and management need a single number: "how healthy is this pump?" We combine power and head residuals into a composite Pump Health Index (PHI) scaled from 0 (critical) to 100 (perfect).

In [ ]:
# Compute Pump Health Index (PHI)
# Approach: normalize each residual by healthy-period standard deviation,
# then combine into a single score.

# Healthy period statistics (reference)
healthy_run = running[running.index < "2024-04-01"]
power_std_healthy = healthy_run["power_residual"].std()
head_std_healthy  = healthy_run["head_residual"].std()

# Normalized absolute residuals (number of healthy-period sigmas)
running["power_zscore"] = np.abs(running["power_residual"]) / max(power_std_healthy, 0.01)
running["head_zscore"]  = np.abs(running["head_residual"]) / max(head_std_healthy, 0.01)

# Combined deviation score (weighted: power gets 60%, head gets 40%)
running["combined_deviation"] = 0.6 * running["power_zscore"] + 0.4 * running["head_zscore"]

# Apply 24h rolling mean for stability
running["combined_deviation_smooth"] = (
    running["combined_deviation"].rolling(24, min_periods=1).mean()
)

# Map to 0-100 health index using a sigmoid-like transform
# At 0 sigma: PHI = 100 (perfect)
# At 3 sigma: PHI ~= 50 (warning)
# At 6+ sigma: PHI -> 0 (critical)
def deviation_to_phi(deviation, scale=3.0):
    """Convert combined deviation to health index 0-100."""
    return 100 * np.exp(-0.5 * (deviation / scale) ** 2)

running["pump_health_index"] = deviation_to_phi(running["combined_deviation_smooth"])

# Plot PHI over time
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(running.index, running["pump_health_index"],
        color="#5FCCDB", linewidth=0.5, alpha=0.6)

# Color-coded rolling mean
phi_smooth = running["pump_health_index"].rolling(48, min_periods=1).mean()
ax.plot(running.index, phi_smooth, color="#2D6A7A", linewidth=2,
        label="48h rolling mean")

# Zone coloring
ax.axhspan(80, 100, alpha=0.05, color="green")
ax.axhspan(50, 80, alpha=0.05, color="yellow")
ax.axhspan(0, 50, alpha=0.05, color="red")
ax.axhline(y=80, color="green", linestyle=":", alpha=0.4, label="Good (>80)")
ax.axhline(y=50, color="orange", linestyle=":", alpha=0.4, label="Warning (<80)")
ax.axhline(y=25, color="red", linestyle=":", alpha=0.4, label="Critical (<50)")

# Shade fault periods
ax.axvspan(pd.Timestamp("2024-04-01"), pd.Timestamp("2024-06-01"),
           alpha=0.1, color="#D69E2E")
ax.axvspan(pd.Timestamp("2024-07-15"), pd.Timestamp("2024-08-21"),
           alpha=0.1, color="#E53E3E")
ax.axvspan(pd.Timestamp("2024-10-15"), pd.Timestamp("2024-12-31"),
           alpha=0.1, color="#718096")

ax.set_xlabel("Date")
ax.set_ylabel("Pump Health Index (0-100)")
ax.set_title("BFP Pump Health Index Over Time")
ax.set_ylim(0, 105)
ax.legend(fontsize=8, loc="lower left")

plt.tight_layout()
plt.show()

# Summary statistics
print("Pump Health Index by period:")
periods_summary = [
    ("Healthy (Jan-Mar)", "2024-01-01", "2024-04-01"),
    ("Seal Fault (Apr-May)", "2024-04-01", "2024-06-01"),
    ("Bearing Fault (Jul-Aug)", "2024-07-15", "2024-08-21"),
    ("Post-Outage (Sep)", "2024-09-15", "2024-10-15"),
    ("Misalignment (Oct-Dec)", "2024-10-15", "2025-01-01"),
]
for label, start, end in periods_summary:
    mask = (running.index >= start) & (running.index < end)
    if mask.sum() > 0:
        phi_vals = running.loc[mask, "pump_health_index"]
        print(f"  {label:30s}: mean={phi_vals.mean():.1f}  min={phi_vals.min():.1f}")

## Step 8: Compare Health Index with Maintenance Events

The final validation: does the health index correlate with actual operator actions? We overlay the operator_actions.csv events on the health index timeline. A good digital twin should show PHI dropping before operators take action, and recovering after maintenance.

In [ ]:
# Overlay maintenance events on health index
fig, ax = plt.subplots(figsize=(10, 5))

phi_smooth = running["pump_health_index"].rolling(48, min_periods=1).mean()
ax.plot(running.index, phi_smooth, color="#2D6A7A", linewidth=2)
ax.set_ylim(0, 105)

# Mark operator actions
event_colors = {
    "START": "#5FCCDB",
    "STOP": "#E53E3E",
    "UNIT_SHUTDOWN": "#718096",
    "UNIT_START": "#D69E2E",
}

for _, event in actions.iterrows():
    color = event_colors.get(event["action"], "gray")
    ax.axvline(x=event["timestamp"], color=color, linestyle="--",
               alpha=0.7, linewidth=1.5)
    ax.annotate(f"{event['equipment']}\n{event['action']}",
                xy=(event["timestamp"], 95),
                fontsize=6, rotation=45, ha="right",
                color=color)

ax.axhline(y=80, color="green", linestyle=":", alpha=0.3)
ax.axhline(y=50, color="orange", linestyle=":", alpha=0.3)
ax.set_xlabel("Date")
ax.set_ylabel("Pump Health Index (0-100)")
ax.set_title("Pump Health Index vs Operator Actions")

plt.tight_layout()
plt.show()

# Analyze PHI at each event
print("\nPHI at operator action times:")
print(f"{'Timestamp':<22s} {'Equipment':<12s} {'Action':<16s} {'PHI':>5s}")
print("-" * 58)
for _, event in actions.iterrows():
    ts = event["timestamp"]
    # Find nearest running hour
    if ts in running.index:
        phi = running.loc[ts, "pump_health_index"]
    else:
        nearest = running.index.get_indexer([ts], method="nearest")[0]
        if nearest >= 0 and nearest < len(running):
            phi = running.iloc[nearest]["pump_health_index"]
        else:
            phi = np.nan
    print(f"{str(ts):<22s} {event['equipment']:<12s} {event['action']:<16s} {phi:>5.1f}")

In [ ]:
# Actual vs Predicted scatter plots for healthy and fault periods
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for ax, name, ylabel in zip(axes,
    ["power", "head"],
    ["Motor Power (kW)", "Discharge Pressure (bar)"]):

    target_col = targets[name]
    pred_col = f"{name}_predicted"

    # Healthy
    h_mask = running.index < "2024-04-01"
    ax.scatter(running.loc[h_mask, target_col],
               running.loc[h_mask, pred_col],
               c="#5FCCDB", s=3, alpha=0.3, label="Healthy")

    # Fault periods
    f_mask = ~h_mask & (running.index < "2024-09-01") | (running.index >= "2024-10-15")
    ax.scatter(running.loc[f_mask, target_col],
               running.loc[f_mask, pred_col],
               c="#E53E3E", s=3, alpha=0.3, label="Fault periods")

    # Perfect prediction line
    lims = [running[target_col].min(), running[target_col].max()]
    ax.plot(lims, lims, color="gray", linestyle="--", alpha=0.5)
    ax.set_xlabel(f"Actual {ylabel}")
    ax.set_ylabel(f"Predicted {ylabel}")
    ax.set_title(f"Actual vs Predicted: {ylabel}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## What You Built and Next Steps

In this guide you built a **physics-informed digital twin** for the SP&L boiler feed pumps:

1. **Loaded OEM pump curves** and created interpolation functions for head, efficiency, and power as a function of flow
2. **Plotted manufacturer performance curves** and identified the Best Efficiency Point (BEP)
3. **Computed expected performance** at each operating point by interpolating the OEM curves
4. **Measured performance deviation** between actual and OEM-expected power, revealing efficiency decay during fault periods
5. **Overlaid actual operating points** on OEM curves, color-coded by fault period, showing how faults shift operating points away from the design curve
6. **Built a physics-informed ML model** trained on healthy baseline data that incorporates OEM expected values as features (Gradient Boosting Regressor)
7. **Monitored residuals** from the physics-informed model, showing them growing during fault periods while staying near zero during healthy operation
8. **Created a Pump Health Index (PHI)** combining multiple residuals into a single 0-100 score using Gaussian-scaled deviation from healthy baseline
9. **Validated against operator actions**, showing the PHI dropping before operators took corrective action and correlating with planned pump swaps and shutdowns

The digital twin approach combines the strengths of physics (OEM curves provide structure) with data (ML learns site-specific corrections). This hybrid approach is more robust than either pure physics models or pure data-driven models alone.

**What makes this a digital twin (not just a model):**
- It maintains a running representation of the physical asset's current state (PHI)
- It compares actual behavior against both design intent (OEM curves) and learned healthy behavior (baseline model)
- It can be continuously updated as new operating data arrives
- It provides actionable output (health score, deviation trends) that maps to maintenance decisions